# 🏙️ Toronto Airbnb Analysis
## Week 1 — Load & Explore

**Business question:** What drives Airbnb pricing in Toronto, and which neighbourhoods offer the best opportunity for hosts?

**This notebook:** First look at the raw data. No cleaning yet — just understanding what we have and what's broken.

**Data source:** [Inside Airbnb](http://insideairbnb.com/get-the-data/) → Toronto → `listings.csv.gz`

---

## 0. Setup
Install dependencies and import libraries. Run this cell first.

In [1]:
# Run this once if you haven't installed pandas yet
# !pip install pandas

import pandas as pd

print("Libraries loaded ✓")

Libraries loaded ✓


## 1. Load the Data

We load the raw CSV directly from the `.gz` file — no need to unzip it manually. Pandas handles it automatically.

> **Make sure** `listings.csv.gz` is in the same folder as this notebook.

In [3]:
df = pd.read_csv('listings.csv.', low_memory=False)

print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

Rows    : 15,776
Columns : 85


## 2. What Columns Do We Have?

Before doing anything, we need to know what fields exist. Some will be useful, most won't.

In [4]:
print(f"All {df.shape[1]} columns:\n")
for col in df.columns:
    print(f"  {col}")

All 85 columns:

  id
  listing_url
  scrape_id
  last_scraped
  source
  name
  description
  neighborhood_overview
  picture_url
  host_id
  host_url
  host_profile_id
  host_profile_url
  host_name
  host_since
  hosts_time_as_user_years
  hosts_time_as_user_months
  hosts_time_as_host_years
  hosts_time_as_host_months
  host_location
  host_about
  host_response_time
  host_response_rate
  host_acceptance_rate
  host_is_superhost
  host_thumbnail_url
  host_picture_url
  host_neighbourhood
  host_listings_count
  host_total_listings_count
  host_verifications
  host_has_profile_pic
  host_identity_verified
  neighbourhood
  neighbourhood_cleansed
  neighbourhood_group_cleansed
  latitude
  longitude
  property_type
  room_type
  accommodates
  bathrooms
  bathrooms_text
  bedrooms
  beds
  amenities
  price
  minimum_nights
  maximum_nights
  minimum_minimum_nights
  maximum_minimum_nights
  minimum_maximum_nights
  maximum_maximum_nights
  minimum_nights_avg_ntm
  maximum_nights_a

## 3. Data Types

This tells us what Pandas *thinks* each column is. Watch for columns that should be numbers but show up as `object` (string). That's a cleaning problem we'll fix in Week 2.

> **Hint:** The `price` column will almost certainly show as `object` — it comes in as `$150.00`, not `150.0`.

In [5]:
print(df.dtypes.to_string())

id                                                int64
listing_url                                      object
scrape_id                                         int64
last_scraped                                     object
source                                           object
name                                             object
description                                      object
neighborhood_overview                           float64
picture_url                                      object
host_id                                           int64
host_url                                         object
host_profile_id                                 float64
host_profile_url                                 object
host_name                                        object
host_since                                      float64
hosts_time_as_user_years                        float64
hosts_time_as_user_months                       float64
hosts_time_as_host_years                        

## 4. Missing Values

Every real dataset has gaps. Here we find which columns have missing data and how bad it is. Columns missing more than 50% of values are usually dropped. Columns missing 5–20% need a decision — drop the rows, or fill with a sensible default.

In [6]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if missing.empty:
    print("No missing values found.")
else:
    print(f"{'Column':<45} {'Missing':>8}  {'% Missing':>10}")
    print("-" * 67)
    for col, count in missing.items():
        pct = (count / len(df)) * 100
        flag = " ⚠️ HIGH" if pct > 40 else ""
        print(f"  {col:<43} {count:>8,}  {pct:>9.1f}%{flag}")

Column                                         Missing   % Missing
-------------------------------------------------------------------
  host_response_rate                            15,776      100.0% ⚠️ HIGH
  host_response_time                            15,776      100.0% ⚠️ HIGH
  neighborhood_overview                         15,776      100.0% ⚠️ HIGH
  neighbourhood_group_cleansed                  15,776      100.0% ⚠️ HIGH
  neighbourhood                                 15,776      100.0% ⚠️ HIGH
  estimated_revenue_l365d                       15,776      100.0% ⚠️ HIGH
  host_total_listings_count                     15,776      100.0% ⚠️ HIGH
  host_neighbourhood                            15,776      100.0% ⚠️ HIGH
  host_thumbnail_url                            15,776      100.0% ⚠️ HIGH
  host_acceptance_rate                          15,776      100.0% ⚠️ HIGH
  price                                         15,776      100.0% ⚠️ HIGH
  calendar_updated                      

## 5. The Price Column Problem

This is intentionally broken — and fixing it will be your first real cleaning task in Week 2.

The `price` column comes in as a string like `"$150.00"`. You can't do math on a string. We need to strip the `$` and `,` characters and convert it to a float.

In [7]:
if 'price' in df.columns:
    print(f"Data type  : {df['price'].dtype}")
    print(f"\nSample values:")
    for val in df['price'].dropna().head(8).values:
        print(f"  {val}")
    print("\n⚠️  This is a string with '$' — you cannot do math on it yet.")
    print("   You'll fix this in Week 2.")

Data type  : float64

Sample values:

⚠️  This is a string with '$' — you cannot do math on it yet.
   You'll fix this in Week 2.


## 6. Preview the Key Columns

Out of all the columns, these are the ones we'll use most throughout the project. Let's see the first few rows of just these.

In [8]:
key_cols = [
    'id', 'name', 'neighbourhood_cleansed', 'room_type',
    'price', 'minimum_nights', 'number_of_reviews',
    'review_scores_rating', 'availability_365', 'host_id'
]

# Only keep columns that actually exist in this dataset
existing = [c for c in key_cols if c in df.columns]

df[existing].head(10)

,id,name,neighbourhood_cleansed,room_type,price,minimum_nights,number_of_reviews,review_scores_rating,availability_365,host_id
0,44452,Yonge & Bloor Studio Skyline,Rosedale-Moore Park,Entire home/apt,NaN,28.0,68,4.19,261,195095
1,44469,Yonge & Bloor 2 Bedroom Apartment,Rosedale-Moore Park,Private room,NaN,30.0,8,4.00,365,195095
2,45399,Fountain View Studio - Eaton center,Bay Street Corridor,Entire home/apt,NaN,28.0,89,4.17,357,195095
3,45893,Yonge & Bloor Lakeview Master BR,Rosedale-Moore Park,Private room,NaN,28.0,24,4.40,365,195095
4,50110,Yorkville one bedroom Condo,Church-Yonge Corridor,Entire home/apt,NaN,28.0,61,4.63,365,195095
5,62545,Furnished Heritage Rooms Downtown 1,Niagara,Private room,NaN,28.0,23,4.85,349,304551
6,64641,Furnished Heritage Rooms Downtown 3,Niagara,Private room,NaN,30.0,28,4.81,365,304551
7,64645,Furnished Downtown Suite with Private Washroom,Niagara,Private room,NaN,30.0,18,4.75,200,304551
8,66856,Furnished Downtown Room,Niagara,Private room,NaN,30.0,5,5.00,229,304551
9,71118,Downtown Loft Suite with Private Washroom,Niagara,Entire home/apt,NaN,30.0,13,4.58,173,304551


## 7. Neighbourhood Breakdown

How many listings does each neighbourhood have? This tells us which areas are most active in the Toronto Airbnb market.

In [9]:
if 'neighbourhood_cleansed' in df.columns:
    neighbourhoods = df['neighbourhood_cleansed'].value_counts()
    print(f"Total unique neighbourhoods: {len(neighbourhoods)}\n")
    print(f"{'Neighbourhood':<40} {'Listings':>10}")
    print("-" * 52)
    for name, count in neighbourhoods.head(15).items():
        print(f"  {name:<38} {count:>10,}")

Total unique neighbourhoods: 140

Neighbourhood                              Listings
----------------------------------------------------
  Waterfront Communities-The Island           2,659
  Niagara                                       585
  Church-Yonge Corridor                         512
  Annex                                         511
  Kensington-Chinatown                          438
  Moss Park                                     420
  Dovercourt-Wallace Emerson-Junction           387
  Bay Street Corridor                           370
  Trinity-Bellwoods                             354
  Willowdale East                               314
  South Riverdale                               290
  Little Portugal                               275
  Palmerston-Little Italy                       235
  York University Heights                       222
  Islington-City Centre West                    200


## 8. Room Type Breakdown

Airbnb has four room types. The split between them affects pricing significantly — entire homes command higher prices than shared rooms.

In [10]:
if 'room_type' in df.columns:
    room_counts = df['room_type'].value_counts()
    print(f"{'Room Type':<35} {'Count':>8}  {'% of Total':>10}")
    print("-" * 57)
    for rtype, count in room_counts.items():
        pct = (count / len(df)) * 100
        bar = "█" * int(pct / 2)
        print(f"  {rtype:<33} {count:>8,}  {pct:>9.1f}%  {bar}")

Room Type                              Count  % of Total
---------------------------------------------------------
  Entire home/apt                     10,575       67.0%  █████████████████████████████████
  Private room                         5,142       32.6%  ████████████████
  Shared room                             38        0.2%  
  Hotel room                              21        0.1%  


## 9. Basic Statistics on Numeric Columns

`describe()` gives us count, mean, min, max, and quartiles for every numeric column. This is a quick way to spot outliers — for example, a `minimum_nights` value of 1000 is obviously wrong.

In [11]:
numeric_cols = [
    'minimum_nights', 'number_of_reviews',
    'review_scores_rating', 'availability_365'
]

existing_numeric = [c for c in numeric_cols if c in df.columns]
df[existing_numeric].describe().round(2)

,minimum_nights,number_of_reviews,review_scores_rating,availability_365
count,15767.00,15776.00,12545.00,15776.00
mean,20.54,34.45,4.80,226.34
std,24.53,66.72,0.38,112.99
min,1.00,0.00,1.00,0.00
25%,2.00,1.00,4.75,127.00
50%,28.00,9.00,4.90,245.00
75%,28.00,40.00,5.00,344.00
max,730.00,1324.00,5.00,365.00


## 10. ✅ Week 1 Homework

Before moving to Week 2, write down your answers to these questions. Open a text file or a new markdown cell below and record what you found. These notes become your project README later.

1. How many listings are in the dataset?
2. Which neighbourhood has the most listings?
3. What percentage of listings are `Entire home/apt`?
4. Name 3 columns with significant missing data — what percentage is missing?
5. Why can't you calculate the average price right now?
6. What's the maximum value of `minimum_nights`? Does it seem realistic?

---

> **Week 2 preview:** You'll fix the price column, drop bad rows, handle missing values, and export a clean CSV. That clean CSV becomes the foundation for every chart and query you build after.

In [12]:
# Your notes go here — add your answers as print statements or just read them from the outputs above
print("My Week 1 findings:")
print(f"  Total listings      : {len(df):,}")
print(f"  Total columns       : {df.shape[1]}")
print(f"  Neighbourhoods      : {df['neighbourhood_cleansed'].nunique() if 'neighbourhood_cleansed' in df.columns else 'N/A'}")
print(f"  Room types          : {df['room_type'].nunique() if 'room_type' in df.columns else 'N/A'}")
print(f"  Price column type   : {df['price'].dtype if 'price' in df.columns else 'N/A'}  ← needs fixing in Week 2")

My Week 1 findings:
  Total listings      : 15,776
  Total columns       : 85
  Neighbourhoods      : 140
  Room types          : 4
  Price column type   : float64  ← needs fixing in Week 2
